# Baseline U-Net34 trên Google Colab — chỉ dùng ISIC 2018 Training

Notebook tự chứa toàn bộ pipeline: tải dữ liệu bằng `kagglehub`, kiểm tra pairing ảnh–mask, chia **Training chính thức** thành 3 fold, chạy LR Finder trên train subset, augmentation chỉ trên train subset và chọn checkpoint BestDice bằng fold holdout sạch.

Notebook **không đọc và không đánh giá** official validation/test. Nó ưu tiên chỉ tải hai thư mục Training; nếu phiên bản `kagglehub` chưa hỗ trợ tải theo thư mục thì sẽ fallback tải cả dataset, nhưng pipeline vẫn chỉ mở file trong Training. Fold holdout chỉ được tách từ Training để chọn checkpoint, không phải official validation.

> Trên Colab chọn `Runtime → Change runtime type → GPU`. Nếu muốn checkpoint còn sau khi runtime bị reset, giữ `USE_GOOGLE_DRIVE=True`; nếu không muốn mount Drive thì đổi thành `False`.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'kagglehub>=1.0.0', 'albumentations>=2,<3',
])
print('Dependencies installed.')

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import time
from pathlib import Path

import albumentations as A
import cv2
import kagglehub
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import DataLoader, Dataset, get_worker_info
from torchvision import models
from tqdm.auto import tqdm

try:
    import psutil
except ImportError:
    psutil = None

# -------------------- Cấu hình baseline --------------------
DATASET_HANDLE = 'tschandl/isic2018-challenge-task1-data-segmentation'
TRAIN_IMAGE_DIRNAME = 'ISIC2018_Task1-2_Training_Input'
TRAIN_MASK_DIRNAME = 'ISIC2018_Task1_Training_GroundTruth'
USE_GOOGLE_DRIVE = True
SEED = 42
N_FOLDS = 3
SIZES = (128, 256)
FROZEN_EPOCHS = 30
UNFROZEN_EPOCHS = 30
PER_GPU_BATCH_BY_SIZE = {128: 16, 256: 8}
NUM_WORKERS = min(2, os.cpu_count() or 1)
PREFETCH_FACTOR = 1
PIN_MEMORY = True
CACHE_RESIZED_IN_RAM = True
SAVE_LAST_EVERY = 5
LRF_ITERS = 50
LR_MIN, LR_MAX, LRF_BETA = 1e-6, 1.0, 0.9
LR_SELECTION_FACTOR = 0.1
MAX_LR_FROZEN, MAX_LR_UNFROZEN = 3e-3, 1e-3
CUT_FRAC, STLR_RATIO = 0.1, 32.0
WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
PRED_THRESHOLD = 0.5
PRETRAINED = True
RESUME = True
FAST_DEV_RUN = False
PROTOCOL_VERSION = 1

if FAST_DEV_RUN:
    SIZES = (128,)
    FROZEN_EPOCHS = UNFROZEN_EPOCHS = 1
    LRF_ITERS = 8

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/isic2018_unet34_colab')
else:
    OUTPUT_ROOT = Path('/content/isic2018_unet34_colab')

RUN_NAME = 'fast_dev_v1' if FAST_DEV_RUN else 'full_v1'
RUN_DIR = OUTPUT_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
LOG_DIR = RUN_DIR / 'logs'
for directory in (RUN_DIR, CHECKPOINT_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

set_seed()
cv2.setNumThreads(0)
torch.set_float32_matmul_precision('high')
if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật GPU trong Colab: Runtime → Change runtime type → GPU.')

DEVICE = torch.device('cuda:0')
GPU_COUNT = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
BATCH_SIZE_BY_SIZE = {size: batch * GPU_COUNT for size, batch in PER_GPU_BATCH_BY_SIZE.items()}
USE_AMP = True
print({
    'device': str(DEVICE), 'gpus': GPU_NAMES, 'global_batch': BATCH_SIZE_BY_SIZE,
    'workers': NUM_WORKERS, 'run_dir': str(RUN_DIR),
})

def memory_status():
    parts = []
    if psutil is not None:
        process = psutil.Process(os.getpid())
        info = process.memory_full_info()
        main_bytes = getattr(info, 'uss', info.rss)
        worker_bytes = 0
        for child in process.children(recursive=True):
            try:
                child_info = child.memory_full_info()
                worker_bytes += getattr(child_info, 'uss', child_info.rss)
            except psutil.Error:
                pass
        parts.extend([f'ram_main={main_bytes / 2**30:.2f}GiB',
                      f'ram_workers={worker_bytes / 2**30:.2f}GiB'])
    allocated = sum(torch.cuda.memory_allocated(i) for i in range(GPU_COUNT))
    reserved = sum(torch.cuda.memory_reserved(i) for i in range(GPU_COUNT))
    parts.extend([f'cuda_alloc={allocated / 2**30:.2f}GiB',
                  f'cuda_reserved={reserved / 2**30:.2f}GiB'])
    return ' '.join(parts)

In [ ]:
def resolve_requested_directory(downloaded_path, directory_name):
    path = Path(downloaded_path)
    if path.is_dir() and path.name == directory_name:
        return path
    direct = path / directory_name
    if direct.is_dir():
        return direct
    matches = sorted(candidate for candidate in path.rglob(directory_name) if candidate.is_dir())
    if len(matches) != 1:
        raise RuntimeError(f'Không tìm thấy duy nhất {directory_name} trong {path}: {matches}')
    return matches[0]

# Ưu tiên chỉ tải hai thư mục Training. Một số bản kagglehub cũ chưa hỗ trợ
# directory path; fallback vẫn chỉ đọc Training dù archive đầy đủ đã được tải.
try:
    training_images_download = kagglehub.dataset_download(
        DATASET_HANDLE, path=TRAIN_IMAGE_DIRNAME
    )
    training_masks_download = kagglehub.dataset_download(
        DATASET_HANDLE, path=TRAIN_MASK_DIRNAME
    )
    TRAINING_ONLY_DOWNLOAD = True
except Exception as directory_download_error:
    print('Directory-only download unavailable; falling back to the full archive:',
          type(directory_download_error).__name__)
    full_download = kagglehub.dataset_download(DATASET_HANDLE)
    training_images_download = full_download
    training_masks_download = full_download
    TRAINING_ONLY_DOWNLOAD = False
TRAIN_IMAGE_DIR = resolve_requested_directory(training_images_download, TRAIN_IMAGE_DIRNAME)
TRAIN_MASK_DIR = resolve_requested_directory(training_masks_download, TRAIN_MASK_DIRNAME)
print('Training images:', TRAIN_IMAGE_DIR)
print('Training masks :', TRAIN_MASK_DIR)

image_paths = sorted(TRAIN_IMAGE_DIR.glob('*.jpg')) + sorted(TRAIN_IMAGE_DIR.glob('*.jpeg'))
mask_paths = sorted(TRAIN_MASK_DIR.glob('*.png'))
images_by_id = {path.stem: path for path in image_paths}
masks_by_id = {}
for path in mask_paths:
    suffix = '_segmentation'
    if not path.stem.lower().endswith(suffix):
        continue
    image_id = path.stem[:-len(suffix)]
    if image_id in masks_by_id:
        raise RuntimeError(f'Mask trùng ID: {image_id}')
    masks_by_id[image_id] = path

missing_masks = sorted(set(images_by_id) - set(masks_by_id))
missing_images = sorted(set(masks_by_id) - set(images_by_id))
if missing_masks or missing_images:
    raise RuntimeError(
        f'Pairing lỗi: thiếu mask={missing_masks[:5]}, thiếu image={missing_images[:5]}'
    )
train_records = [
    {'image_id': image_id, 'image_path': images_by_id[image_id],
     'mask_path': masks_by_id[image_id]}
    for image_id in sorted(images_by_id)
]
if len(train_records) != 2594:
    raise RuntimeError(f'Mong đợi 2594 cặp Training, tìm thấy {len(train_records)}')

generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(train_records), generator=generator).tolist()
fold_sizes = [len(train_records) // N_FOLDS] * N_FOLDS
for index in range(len(train_records) % N_FOLDS):
    fold_sizes[index] += 1
fold_indices, cursor = [], 0
for fold_size in fold_sizes:
    fold_indices.append(permutation[cursor:cursor + fold_size])
    cursor += fold_size

folds = []
all_indices = set(range(len(train_records)))
for holdout_indices in fold_indices:
    holdout_set = set(holdout_indices)
    train_indices = sorted(all_indices - holdout_set)
    train_subset = [train_records[i] for i in train_indices]
    holdout_subset = [train_records[i] for i in holdout_indices]
    assert set(r['image_id'] for r in train_subset).isdisjoint(
        r['image_id'] for r in holdout_subset
    )
    folds.append((train_subset, holdout_subset))

print('Training pairs:', len(train_records), 'fold sizes:', fold_sizes)
print('Training-only download:', TRAINING_ONLY_DOWNLOAD)
print('Official validation/test are never opened or evaluated.')

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
RAM_CACHE = {}

def gray_world(image, **kwargs):
    del kwargs
    image_float = image.astype(np.float32)
    channel_means = image_float.mean(axis=(0, 1))
    gray_mean = channel_means.mean()
    scale = gray_mean / np.maximum(channel_means, 1e-6)
    return np.clip(image_float * scale.reshape(1, 1, 3), 0, 255).astype(np.uint8)

def build_transform(size, train, seed, preprocessed=False):
    steps = []
    if not preprocessed:
        steps.extend([
            A.Resize(size, size, interpolation=cv2.INTER_LINEAR,
                     mask_interpolation=cv2.INTER_NEAREST),
            A.Lambda(image=gray_world, p=1.0),
        ])
    if train:
        steps.extend([
            A.D4(p=1.0),
            A.Affine(rotate=(-44, 44), scale=(1.0, 1.05),
                     interpolation=cv2.INTER_LINEAR,
                     mask_interpolation=cv2.INTER_NEAREST, p=0.75),
            A.RandomBrightnessContrast(brightness_limit=0.2,
                                       contrast_limit=0.2, p=0.75),
        ])
    steps.append(ToTensorV2())
    return A.Compose(steps, seed=int(seed))

class ISICDataset(Dataset):
    def __init__(self, records, size, train, seed=SEED):
        if not records:
            raise ValueError('Dataset rỗng')
        self.records = list(records)
        self.size = int(size)
        self.train = bool(train)
        cache = RAM_CACHE.get(self.size)
        indices = None if cache is None else [cache['id_to_index'].get(r['image_id'])
                                              for r in self.records]
        self.cache = cache if indices is not None and all(i is not None for i in indices) else None
        self.cache_indices = indices if self.cache is not None else None
        self.transform = build_transform(
            self.size, self.train, seed, preprocessed=self.cache is not None
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        if self.cache is not None:
            cache_index = self.cache_indices[index]
            image_np = self.cache['images'][cache_index].copy()
            mask_np = self.cache['masks'][cache_index].copy()
        else:
            with Image.open(record['image_path']) as source:
                image_np = np.asarray(source.convert('RGB'), dtype=np.uint8).copy()
            with Image.open(record['mask_path']) as source:
                mask_np = np.asarray(source.convert('L'), dtype=np.uint8).copy()
            if image_np.shape[:2] != mask_np.shape:
                raise ValueError(f"Image/mask lệch kích thước: {record['image_id']}")
            mask_np = (mask_np > 127).astype(np.uint8)
        transformed = self.transform(image=image_np, mask=mask_np)
        image, mask = transformed['image'], transformed['mask']
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        return image.contiguous(), (mask > 0).float().contiguous()

def seed_worker(worker_id):
    del worker_id
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    worker = get_worker_info()
    if worker is not None and hasattr(worker.dataset.transform, 'set_random_seed'):
        worker.dataset.transform.set_random_seed(worker_seed)

def make_loader(records, size, train, seed, batch_size=None, drop_last=False):
    if batch_size is None:
        batch_size = BATCH_SIZE_BY_SIZE[int(size)]
    kwargs = dict(
        dataset=ISICDataset(records, size, train, seed),
        batch_size=int(batch_size), shuffle=bool(train), drop_last=bool(drop_last),
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=False,
        worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(int(seed)),
    )
    if NUM_WORKERS > 0:
        kwargs['prefetch_factor'] = PREFETCH_FACTOR
    return DataLoader(**kwargs)

def build_ram_cache(records, size):
    images = np.empty((len(records), size, size, 3), dtype=np.uint8)
    masks = np.empty((len(records), size, size), dtype=np.uint8)
    id_to_index = {}
    for index, record in enumerate(tqdm(records, desc=f'RAM cache {size}px')):
        with Image.open(record['image_path']) as source:
            image_np = np.asarray(source.convert('RGB'), dtype=np.uint8).copy()
        with Image.open(record['mask_path']) as source:
            mask_np = np.asarray(source.convert('L'), dtype=np.uint8).copy()
        if image_np.shape[:2] != mask_np.shape:
            raise ValueError(f"Image/mask lệch kích thước: {record['image_id']}")
        resized = cv2.resize(image_np, (size, size), interpolation=cv2.INTER_LINEAR)
        images[index] = gray_world(resized)
        binary_mask = (mask_np > 127).astype(np.uint8)
        masks[index] = cv2.resize(binary_mask, (size, size),
                                  interpolation=cv2.INTER_NEAREST)
        id_to_index[record['image_id']] = index
    return {'images': images, 'masks': masks, 'id_to_index': id_to_index}

if CACHE_RESIZED_IN_RAM:
    for cache_size in SIZES:
        RAM_CACHE[cache_size] = build_ram_cache(train_records, cache_size)
    cache_gib = sum(v['images'].nbytes + v['masks'].nbytes
                    for v in RAM_CACHE.values()) / 2**30
    print(f'Resized uint8 cache: {cache_gib:.2f} GiB')

clean_dataset = ISICDataset(train_records[:3], SIZES[0], train=False)
for index in range(len(clean_dataset)):
    x1, y1 = clean_dataset[index]
    x2, y2 = clean_dataset[index]
    assert torch.equal(x1, x2) and torch.equal(y1, y2)
    assert x1.dtype == torch.uint8 and y1.dtype == torch.float32
    assert set(torch.unique(y1).tolist()).issubset({0.0, 1.0})
    assert 0.0 < float(y1.mean()) < 1.0
del clean_dataset, x1, x2, y1, y2
print('Dataset, mask and clean-transform checks: OK')

In [ ]:
class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels // 2, 2, stride=2)
        self.skip_projection = nn.Conv2d(skip_channels, out_channels // 2, 1)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        skip = self.skip_projection(skip)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='nearest')
        return self.bn(F.relu(torch.cat([x, skip], dim=1)))

class UNet34(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        try:
            backbone = models.resnet34(weights=weights)
        except Exception as error:
            raise RuntimeError(
                'Không tải được ResNet34 ImageNet weights; kiểm tra kết nối Colab.'
            ) from error
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)
        self.pool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.up4 = UpBlock(512, 256, 256)
        self.up3 = UpBlock(256, 128, 256)
        self.up2 = UpBlock(256, 64, 256)
        self.up1 = UpBlock(256, 64, 256)
        self.up0 = UpBlock(256, 3, 16)
        self.head = nn.Conv2d(16, 1, 1)

    @property
    def first_layer_group(self):
        return (self.stem, self.layer1, self.layer2)

    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.layer1(self.pool(s0))
        s2 = self.layer2(s1)
        s3 = self.layer3(s2)
        bottleneck = self.layer4(s3)
        decoded = self.up4(bottleneck, s3)
        decoded = self.up3(decoded, s2)
        decoded = self.up2(decoded, s1)
        decoded = self.up1(decoded, s0)
        decoded = self.up0(decoded, x)
        return self.head(decoded)

def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def to_runtime(model):
    model = model.to(DEVICE, memory_format=torch.channels_last)
    if GPU_COUNT > 1:
        model = nn.DataParallel(model, device_ids=list(range(GPU_COUNT)))
    return model

def freeze_first_layer_group(model):
    base = unwrap_model(model)
    for parameter in base.parameters():
        parameter.requires_grad = True
    for module in base.first_layer_group:
        for child in module.modules():
            for parameter in child.parameters(recurse=False):
                parameter.requires_grad = isinstance(child, nn.BatchNorm2d)

def unfreeze_with_frozen_bn(model):
    for parameter in model.parameters():
        parameter.requires_grad = True
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            for parameter in module.parameters():
                parameter.requires_grad = False

def set_train_mode(model, freeze_bn):
    model.train()
    if freeze_bn:
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()

def cpu_state_dict(model):
    return {key: value.detach().cpu().clone()
            for key, value in unwrap_model(model).state_dict().items()}

@torch.no_grad()
def refresh_cpu_state_dict(model, destination):
    source = unwrap_model(model).state_dict()
    if tuple(source) != tuple(destination):
        raise RuntimeError('state_dict keys changed during training')
    for key, value in source.items():
        destination[key].copy_(value.detach(), non_blocking=False)

def load_model_state(model, state):
    while state and all(key.startswith('module.') for key in state):
        state = {key.removeprefix('module.'): value for key, value in state.items()}
    unwrap_model(model).load_state_dict(state)

shape_model = UNet34(pretrained=False).eval()
with torch.inference_mode():
    shape_output = shape_model(torch.randn(2, 3, 128, 128))
assert shape_output.shape == (2, 1, 128, 128)
assert not any(key.startswith('first_layer_group.') for key in shape_model.state_dict())
print(f'Parameters: {sum(p.numel() for p in shape_model.parameters()) / 1e6:.3f}M')
del shape_model, shape_output

In [ ]:
criterion = nn.BCEWithLogitsLoss()
INPUT_DTYPE = torch.float16
GPU_MEAN = torch.tensor(IMAGENET_MEAN, device=DEVICE, dtype=INPUT_DTYPE).view(1, 3, 1, 1)
GPU_STD = torch.tensor(IMAGENET_STD, device=DEVICE, dtype=INPUT_DTYPE).view(1, 3, 1, 1)

def prepare_inputs(inputs):
    inputs = inputs.to(DEVICE, non_blocking=True).to(dtype=INPUT_DTYPE)
    inputs.div_(255.0).sub_(GPU_MEAN).div_(GPU_STD)
    return inputs.contiguous(memory_format=torch.channels_last)

def per_image_dice(probability, target, threshold=PRED_THRESHOLD):
    prediction = (probability > threshold).float()
    dims = (1, 2, 3)
    intersection = (prediction * target).sum(dims)
    denominator = prediction.sum(dims) + target.sum(dims)
    return torch.where(denominator > 0, 2 * intersection / denominator.clamp(min=1),
                       torch.ones_like(denominator))

@torch.inference_mode()
def evaluate_dice(model, loader):
    model.eval()
    dice_sum = torch.zeros((), device=DEVICE)
    sample_count = 0
    for inputs, targets in loader:
        inputs = prepare_inputs(inputs)
        targets = targets.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            probability = torch.sigmoid(model(inputs).float())
        if not bool(torch.isfinite(probability).all()):
            raise FloatingPointError('Model sinh xác suất NaN/Inf')
        dice_sum += per_image_dice(probability, targets).sum()
        sample_count += targets.shape[0]
    if sample_count == 0:
        raise RuntimeError('Holdout loader rỗng')
    return float((dice_sum / sample_count).cpu())

metric_target = torch.zeros(2, 1, 8, 8, device=DEVICE)
metric_target[0, :, :4] = 1
metric_target[1, :, :, :4] = 1
metric_probability = metric_target.clone()
metric_probability[1] = 0
metric_probability[1, :, :, :2] = 1
assert torch.allclose(per_image_dice(metric_probability, metric_target),
                      torch.tensor([1.0, 2 / 3], device=DEVICE), atol=1e-6)
del metric_target, metric_probability
print('Dice metric check: OK')

In [ ]:
def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)

def trusted_torch_load(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def make_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=USE_AMP)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def trainable_parameters(model):
    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not parameters:
        raise RuntimeError('Không có parameter trainable')
    return parameters

def optimizer_for(model, learning_rate):
    parameters = trainable_parameters(model)
    try:
        return torch.optim.Adam(parameters, lr=learning_rate,
                                weight_decay=WEIGHT_DECAY, fused=True)
    except (TypeError, RuntimeError):
        return torch.optim.Adam(parameters, lr=learning_rate,
                                weight_decay=WEIGHT_DECAY)

def train_batch(model, optimizer, scaler, inputs, targets):
    inputs = prepare_inputs(inputs)
    targets = targets.to(DEVICE, non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
        loss = criterion(model(inputs).float(), targets)
    if not bool(torch.isfinite(loss)):
        raise FloatingPointError('Train loss là NaN/Inf')
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(
        (p for group in optimizer.param_groups for p in group['params']),
        GRAD_CLIP_NORM,
    )
    scaler.step(optimizer)
    scaler.update()
    return loss.detach()

def lr_find(model, records, size, stage_seed, freeze_bn):
    set_seed(stage_seed)
    original_state = cpu_state_dict(model)
    optimizer = optimizer_for(model, LR_MIN)
    scaler = make_scaler()
    loader = make_loader(records, size, train=True, seed=stage_seed,
                         batch_size=BATCH_SIZE_BY_SIZE[size], drop_last=False)
    iterator = iter(loader)
    multiplier = (LR_MAX / LR_MIN) ** (1 / max(1, LRF_ITERS - 1))
    learning_rates, losses = [], []
    moving_average, best_loss = 0.0, float('inf')
    for iteration in range(LRF_ITERS):
        try:
            inputs, targets = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            inputs, targets = next(iterator)
        learning_rate = LR_MIN * multiplier ** iteration
        for group in optimizer.param_groups:
            group['lr'] = learning_rate
        set_train_mode(model, freeze_bn)
        try:
            raw_loss = float(train_batch(model, optimizer, scaler, inputs, targets).cpu())
        except FloatingPointError:
            break
        moving_average = LRF_BETA * moving_average + (1 - LRF_BETA) * raw_loss
        smoothed = moving_average / (1 - LRF_BETA ** (iteration + 1))
        if not np.isfinite(smoothed) or (iteration > 5 and smoothed > 4 * best_loss):
            break
        best_loss = min(best_loss, smoothed)
        learning_rates.append(learning_rate)
        losses.append(smoothed)
    del iterator, loader, optimizer, scaler, inputs, targets
    load_model_state(model, original_state)
    del original_state
    gc.collect()
    set_seed(stage_seed)
    learning_rates = np.asarray(learning_rates, dtype=float)
    losses = np.asarray(losses, dtype=float)
    if len(learning_rates) < 3:
        raise RuntimeError('LR Finder không đủ điểm hợp lệ')
    gradient = np.gradient(losses, np.log10(learning_rates))
    steepest_lr = float(learning_rates[int(np.argmin(gradient))])
    min_loss_lr = float(learning_rates[int(np.argmin(losses))])
    cap = MAX_LR_UNFROZEN if freeze_bn else MAX_LR_FROZEN
    selected_lr = max(LR_MIN, min(steepest_lr * LR_SELECTION_FACTOR,
                                  min_loss_lr / 10, cap))
    print(f'LR Finder {size}px: selected={selected_lr:.2e}')
    return selected_lr, learning_rates.tolist(), losses.tolist()

def stlr_lambda(total_steps, cut_frac=CUT_FRAC, ratio=STLR_RATIO):
    cut = max(1, int(total_steps * cut_frac))
    def schedule(step):
        if step < cut:
            progress = step / cut
        else:
            denominator = cut * (1 / cut_frac - 1)
            progress = max(0.0, 1 - (step - cut) / max(1, denominator))
        return (1 + progress * (ratio - 1)) / ratio
    return schedule

def ids_digest(records):
    text = '\n'.join(record['image_id'] for record in records)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def save_best_state(best_state, best_path, metadata):
    atomic_torch_save({'model_state': best_state, 'meta': metadata}, best_path)

def run_stage(model, train_subset, holdout_subset, size, epochs, freeze_bn,
              fold_number, stage_name, stage_seed):
    stage_dir = CHECKPOINT_DIR / f'fold_{fold_number}' / f'{size}px_{stage_name}'
    best_path = stage_dir / 'best.pt'
    last_path = stage_dir / 'last.pt'
    contract = {
        'protocol_version': PROTOCOL_VERSION, 'fold': fold_number, 'size': size,
        'stage': stage_name, 'epochs': epochs, 'stage_seed': stage_seed,
        'freeze_bn': freeze_bn, 'batch_size': BATCH_SIZE_BY_SIZE[size],
        'train_ids': ids_digest(train_subset), 'holdout_ids': ids_digest(holdout_subset),
        'selection_metric': 'internal_fold_holdout_dice',
    }

    saved = None
    if RESUME and last_path.is_file():
        saved = trusted_torch_load(last_path, map_location='cpu')
        if saved.get('contract') != contract:
            raise RuntimeError(f'Checkpoint không khớp config/split: {last_path}')
        if saved.get('complete'):
            best_payload = trusted_torch_load(best_path, map_location='cpu')
            if best_payload.get('meta', {}).get('contract') != contract:
                raise RuntimeError(f'Best checkpoint không khớp: {best_path}')
            load_model_state(model, best_payload['model_state'])
            print(f'[resume] fold={fold_number} {size}px {stage_name}: complete')
            return saved['result']

    holdout_loader = make_loader(holdout_subset, size, train=False, seed=stage_seed)
    if saved is None:
        selected_lr, lrf_lrs, lrf_losses = lr_find(
            model, train_subset, size, stage_seed, freeze_bn
        )
        initial_dice = evaluate_dice(model, holdout_loader)
        best_score, best_epoch = initial_dice, 0
        history = []
        best_state = cpu_state_dict(model)
        best_meta = {
            'epoch': best_epoch, 'dice': best_score, 'selected_lr': selected_lr,
            'contract': contract,
        }
        save_best_state(best_state, best_path, best_meta)
        start_epoch = 0
    else:
        selected_lr = float(saved['selected_lr'])
        lrf_lrs, lrf_losses = saved['lrf_lrs'], saved['lrf_losses']
        best_score = float(saved['best_score'])
        best_epoch = int(saved['best_epoch'])
        history = list(saved['history'])
        best_payload = trusted_torch_load(best_path, map_location='cpu')
        best_meta = best_payload.get('meta', {})
        if best_meta.get('contract') != contract:
            raise RuntimeError(f'Best checkpoint không khớp: {best_path}')
        if int(best_meta['epoch']) >= best_epoch:
            best_epoch = int(best_meta['epoch'])
            best_score = float(best_meta['dice'])
        best_state = best_payload['model_state']
        del best_payload
        start_epoch = int(saved['next_epoch'])
        load_model_state(model, saved['model_state'])
        print(f'[resume] fold={fold_number} {size}px {stage_name}: epoch {start_epoch + 1}')

    train_loader = make_loader(train_subset, size, train=True,
                               seed=stage_seed + start_epoch, drop_last=False)
    steps_per_epoch = len(train_loader)
    optimizer = optimizer_for(model, selected_lr)
    scaler = make_scaler()
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, stlr_lambda(epochs * steps_per_epoch)
    )
    if saved is not None:
        optimizer.load_state_dict(saved['optimizer_state'])
        scheduler.load_state_dict(saved['scheduler_state'])
        scaler.load_state_dict(saved['scaler_state'])
        saved = None
        gc.collect()

    for epoch in range(start_epoch, epochs):
        epoch_started = time.perf_counter()
        set_train_mode(model, freeze_bn)
        loss_sum = torch.zeros((), device=DEVICE)
        samples_seen = 0
        for inputs, targets in train_loader:
            loss_sum += train_batch(model, optimizer, scaler, inputs, targets)
            samples_seen += inputs.shape[0]
            scheduler.step()
        del inputs, targets
        train_loss = float((loss_sum / steps_per_epoch).cpu())
        holdout_dice = evaluate_dice(model, holdout_loader)
        row = {
            'fold': fold_number, 'size': size, 'stage': stage_name,
            'epoch': epoch + 1, 'train_loss': train_loss,
            'holdout_dice': holdout_dice, 'lr': optimizer.param_groups[0]['lr'],
        }
        history.append(row)
        if holdout_dice > best_score:
            best_score, best_epoch = holdout_dice, epoch + 1
            refresh_cpu_state_dict(model, best_state)
        if (epoch + 1) % SAVE_LAST_EVERY == 0 or epoch + 1 == epochs:
            best_meta = {
                'epoch': best_epoch, 'dice': best_score, 'selected_lr': selected_lr,
                'contract': contract,
            }
            save_best_state(best_state, best_path, best_meta)
            progress = {
                'complete': False, 'contract': contract, 'next_epoch': epoch + 1,
                'selected_lr': selected_lr, 'lrf_lrs': lrf_lrs,
                'lrf_losses': lrf_losses, 'best_score': best_score,
                'best_epoch': best_epoch, 'history': history,
                'model_state': unwrap_model(model).state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state': scaler.state_dict(),
            }
            atomic_torch_save(progress, last_path)
            del progress
        seconds = time.perf_counter() - epoch_started
        print(
            f'[fold {fold_number} | {size}px | {stage_name}] '
            f'epoch {epoch + 1:02d}/{epochs} loss={train_loss:.4f} '
            f'holdout_dice={holdout_dice:.4f} best={best_score:.4f} '
            f'time={seconds:.1f}s img/s={samples_seen / seconds:.1f} {memory_status()}'
        )

    load_model_state(model, best_state)
    result = {
        'fold': fold_number, 'size': size, 'stage': stage_name,
        'selected_lr': selected_lr, 'best_epoch': best_epoch,
        'best_holdout_dice': best_score, 'history': history,
        'lrf_lrs': lrf_lrs, 'lrf_losses': lrf_losses,
        'best_path': str(best_path),
    }
    atomic_torch_save({'complete': True, 'contract': contract, 'result': result}, last_path)
    del best_state, train_loader, holdout_loader, optimizer, scaler, scheduler
    gc.collect()
    return result

In [ ]:
def preflight_batch(size):
    batch_size = BATCH_SIZE_BY_SIZE[size]
    model = to_runtime(UNet34(pretrained=False))
    optimizer = optimizer_for(model, 1e-4)
    scaler = make_scaler()
    inputs = torch.randint(0, 256, (batch_size, 3, size, size), dtype=torch.uint8)
    targets = torch.randint(0, 2, (batch_size, 1, size, size), dtype=torch.float32)
    try:
        loss = float(train_batch(model, optimizer, scaler, inputs, targets).cpu())
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError(
            f'Batch {batch_size} không vừa GPU ở {size}px; giảm PER_GPU_BATCH_BY_SIZE.'
        ) from error
    finally:
        del model, optimizer, scaler, inputs, targets
        gc.collect()
        torch.cuda.empty_cache()
    print(f'Preflight {size}px batch={batch_size}: loss={loss:.4f}')

for size in SIZES:
    preflight_batch(size)

In [ ]:
RUN_CONFIG = {
    'dataset_handle': DATASET_HANDLE, 'official_splits_used': ['Training'],
    'training_only_download': TRAINING_ONLY_DOWNLOAD,
    'official_validation_evaluated': False, 'official_test_evaluated': False,
    'seed': SEED, 'n_folds': N_FOLDS, 'sizes': list(SIZES),
    'frozen_epochs': FROZEN_EPOCHS, 'unfrozen_epochs': UNFROZEN_EPOCHS,
    'batch_size_by_size': BATCH_SIZE_BY_SIZE, 'num_workers': NUM_WORKERS,
    'pretrained': PRETRAINED, 'selection_metric': 'internal_fold_holdout_dice',
    'augmentation_only_on_train_subset': True,
    'protocol_version': PROTOCOL_VERSION,
}
(RUN_DIR / 'config.json').write_text(
    json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False), encoding='utf-8'
)

def train_fold(fold_number, train_subset, holdout_subset):
    fold_seed = SEED + fold_number - 1
    set_seed(fold_seed)
    first_marker = CHECKPOINT_DIR / f'fold_{fold_number}' / f'{SIZES[0]}px_frozen' / 'last.pt'
    continuing = RESUME and first_marker.is_file()
    model = to_runtime(UNet34(pretrained=PRETRAINED and not continuing))
    stage_results = []
    for size_index, size in enumerate(SIZES):
        freeze_first_layer_group(model)
        stage_results.append(run_stage(
            model, train_subset, holdout_subset, size, FROZEN_EPOCHS, False,
            fold_number, 'frozen', fold_seed + size_index * 1000,
        ))
        unfreeze_with_frozen_bn(model)
        stage_results.append(run_stage(
            model, train_subset, holdout_subset, size, UNFROZEN_EPOCHS, True,
            fold_number, 'unfrozen', fold_seed + size_index * 1000 + 500,
        ))

    holdout_loader = make_loader(holdout_subset, SIZES[-1], train=False, seed=fold_seed)
    final_holdout_dice = evaluate_dice(model, holdout_loader)
    final_path = CHECKPOINT_DIR / f'fold_{fold_number}' / 'unet34_best.pt'
    metadata = {
        'fold': fold_number, 'fold_seed': fold_seed, 'config': RUN_CONFIG,
        'train_count': len(train_subset), 'holdout_count': len(holdout_subset),
        'train_ids': ids_digest(train_subset), 'holdout_ids': ids_digest(holdout_subset),
        'holdout_dice': final_holdout_dice, 'stage_results': stage_results,
    }
    atomic_torch_save({'model_state': unwrap_model(model).state_dict(),
                       'meta': metadata}, final_path)
    rows = [row for result in stage_results for row in result['history']]
    pd.DataFrame(rows).to_csv(LOG_DIR / f'fold_{fold_number}_history.csv', index=False)
    del model, holdout_loader
    gc.collect()
    torch.cuda.empty_cache()
    return final_path, metadata

fold_checkpoint_paths, fold_metadata = [], []
training_started = time.time()
for fold_number, (train_subset, holdout_subset) in enumerate(folds, start=1):
    final_path = CHECKPOINT_DIR / f'fold_{fold_number}' / 'unet34_best.pt'
    if RESUME and final_path.is_file():
        payload = trusted_torch_load(final_path, map_location='cpu')
        metadata = payload['meta']
        if (metadata.get('config') != RUN_CONFIG
                or metadata.get('train_ids') != ids_digest(train_subset)
                or metadata.get('holdout_ids') != ids_digest(holdout_subset)):
            raise RuntimeError(f'Fold checkpoint không khớp: {final_path}')
        del payload
        fold_path = final_path
        print(f"[resume] fold {fold_number} complete, Dice={metadata['holdout_dice']:.4f}")
    else:
        print(f'\n===== FOLD {fold_number}/{N_FOLDS}: train={len(train_subset)}, '
              f'holdout={len(holdout_subset)} =====')
        fold_path, metadata = train_fold(fold_number, train_subset, holdout_subset)
    fold_checkpoint_paths.append(str(fold_path))
    fold_metadata.append(metadata)

fold_table = pd.DataFrame([
    {'fold': metadata['fold'], 'train_count': metadata['train_count'],
     'holdout_count': metadata['holdout_count'],
     'internal_holdout_dice': metadata['holdout_dice']}
    for metadata in fold_metadata
])
fold_table.to_csv(LOG_DIR / 'fold_holdout_dice.csv', index=False)
summary = {
    'config': RUN_CONFIG, 'training_samples': len(train_records),
    'elapsed_hours': (time.time() - training_started) / 3600,
    'fold_checkpoints': fold_checkpoint_paths,
    'fold_internal_holdout': fold_table.to_dict(orient='records'),
    'official_validation': None, 'official_test': None,
}
(RUN_DIR / 'training_summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)
display(fold_table.style.format({'internal_holdout_dice': '{:.4f}'}))
print('Training complete. No official validation/test evaluation was run.')
print('Checkpoints:', fold_checkpoint_paths)
print('Summary:', RUN_DIR / 'training_summary.json')

## Kết quả được tạo

- Ba checkpoint `fold_1..3/unet34_best.pt`, mỗi checkpoint được chọn bằng Dice trên holdout nội bộ của chính tập Training.
- Lịch sử loss/Dice từng epoch trong thư mục `logs`.
- `training_summary.json` ghi rõ official validation và test đều là `null`.
- Notebook không tạo dự đoán và không thực hiện ensemble evaluation; đây là notebook **train-only**.